# Loops, Control Flow & Log Processing

## What you'll practice

Processing a stream of lines with state: counting, filtering, merging multi-line entries, sliding windows.
You'll use `for` over iterables directly (no index loops), `continue` for guard clauses, `dict.get` counters,
`collections.deque` for windows, and `datetime` for time math.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/fundamentals/test_01_03_loops_and_log_processing.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/01_python_fundamentals/03_loops_and_log_processing.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
SAMPLE_LOG = LAB / "datasets/logs/app_small.log"   # 250 real-looking lines
lines = SAMPLE_LOG.read_text(encoding="utf-8").splitlines()
print(len(lines), "lines; first:", lines[0][:90])

## Exercise 1 — 🟢 Easy · Count log levels

**Implement:** `count_levels(lines: list[str]) -> dict[str, int]`

The level is the 2nd whitespace-separated token. Count lines per level, considering only lines whose
first token starts with a 4-digit year (`line[:4].isdigit()`) and that have at least 2 tokens.

**Input:** `lines: list[str]`

**Expected output:** `dict[str, int]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
count_levels(["2026-03-01T00:00:01Z INFO [x] ...", "", "Traceback ...", "2026-03-01T00:00:02Z ERROR [y] ..."])
# {'INFO': 1, 'ERROR': 1}
```

**Check:** `pytest tests/fundamentals/test_01_03_loops_and_log_processing.py -k "test_count_levels or test_count_levels_empty"`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Find slow requests

**Implement:** `find_slow_requests(lines: list[str], threshold_ms: int) -> list[tuple[str, int]]`

Return `(req_id, latency_ms)` for every line with `latency_ms` strictly greater than `threshold_ms`, preserving order.
Find tokens by key (`req=`, `latency_ms=`) rather than by position. Skip lines missing either key or with a
non-integer latency.

**Input:** `lines: list[str], threshold_ms: int`

**Expected output:** `list[tuple[str, int]]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
find_slow_requests(['... req=a1 ... latency_ms=1500 msg="x"', '... req=b2 ... latency_ms=20'], 1000)
# [('a1', 1500)]
```

**Check:** `pytest tests/fundamentals/test_01_03_loops_and_log_processing.py -k "test_find_slow_requests or test_find_slow_requests_none"`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Merge multi-line log entries

**Implement:** `merge_multiline(lines: list[str]) -> list[str]`

Tracebacks span several lines. A line that starts with a timestamp (`^\d{4}-\d{2}-\d{2}T`) begins a new entry;
any other non-blank line is a continuation and is appended to the current entry with `"\n"`. Continuations before
the first entry are dropped. Blank lines are ignored.

**Input:** `lines: list[str]`

**Expected output:** `list[str]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
merge_multiline(["2026-03-01T00:00:01Z ERROR boom", "Traceback (most recent call last):", "ValueError: x",
                 "2026-03-01T00:00:02Z INFO ok"])
# ['2026-03-01T00:00:01Z ERROR boom\nTraceback (most recent call last):\nValueError: x', '2026-03-01T00:00:02Z INFO ok']
```

**Check:** `pytest tests/fundamentals/test_01_03_loops_and_log_processing.py -k "test_merge_multiline or test_merge_multiline_empty"`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Sessionize user activity

**Implement:** `count_sessions(events: list[tuple[str, datetime]], gap_minutes: int = 30) -> dict[str, int]`

Events are `(user_id, timestamp)` tuples **not necessarily sorted**. A new session starts for a user when the gap
since that user's previous event is **more than** `gap_minutes`. Return sessions per user.

**Input:** `events: list[tuple[str, datetime]], gap_minutes: int = 30`

**Expected output:** `dict[str, int]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
from datetime import datetime as dt
count_sessions([("u1", dt(2026,1,1,10,0)), ("u1", dt(2026,1,1,10,20)), ("u1", dt(2026,1,1,11,0)), ("u2", dt(2026,1,1,9,0))])
# {'u1': 2, 'u2': 1}
```

**Check:** `pytest tests/fundamentals/test_01_03_loops_and_log_processing.py -k "test_count_sessions or test_count_sessions_boundary or test_count_sessions_empty"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Error-burst alerting with a sliding window

**Implement:** `error_bursts(entries: list[tuple[datetime, str]], window_seconds: int = 60, threshold: int = 5) -> list[datetime]`

`entries` are `(timestamp, level)` in time order. Fire an alert when the number of `ERROR`/`CRITICAL` entries in the
window `(ts - window_seconds, ts]` reaches `threshold`. Return the timestamps at which alerts fire. After an alert,
**reset** the window (start counting from zero) so one burst produces one alert.

Then run it on the parsed `app.log` (use your `parse_log_line` idea) and see how many bursts exist with
`window_seconds=300, threshold=5`.

**Input:** `entries: list[tuple[datetime, str]], window_seconds: int = 60, threshold: int = 5`

**Expected output:** `list[datetime]`

**Constraints:** Use `collections.deque` — popping from the left of a list is O(n).

**Example:** the tests are executable examples — read `test_error_bursts_fires_once_per_burst, test_error_bursts_window_edge, test_error_bursts_ignores_info_and_counts_critical` in `tests/fundamentals/test_01_03_loops_and_log_processing.py`.

**Check:** `pytest tests/fundamentals/test_01_03_loops_and_log_processing.py -k "test_error_bursts_fires_once_per_burst or test_error_bursts_window_edge or test_error_bursts_ignores_info_and_counts_critical or test_error_bursts_large_input"`

**YOUR CODE ↓**